# FiberBrowser Jupyter Client Demo

This notebook demonstrates how to interact with a running FiberBrowser instance from Jupyter.

**Prerequisites:** install FiberBrowser's packages (`pip install -r requirements.txt` in the repository root; this includes `requests`, which the client uses) and start FiberBrowser first:
```bash
python browser.py -b output/sample_footprints.bam -g genes.gtf
```
Run this notebook from the repository's `examples/` folder (the first cell puts the repository root on `sys.path` so `browser_client` imports).

A local FiberBrowser requires its per-launch access token; run as the same user as the server, `BrowserClient` finds it in `~/.fiberhmm/browser_runtime.json`. Otherwise pass the link the server printed: `BrowserClient("http://localhost:8000/?token=...")`.

## 1. Connect to the browser

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))  # the repository root

from browser_client import BrowserClient

client = BrowserClient("http://localhost:8000")
print(client)

## 2. Navigate

In [ ]:
# Jump to a specific region
client.goto("chrI", 10000, 15000)

# Check where we are
client.get_region()

In [ ]:
# Or navigate by gene name (requires GTF loaded)
# client.goto_gene("YAL038W", padding=2000)

## 3. Get reads

In [ ]:
# Get reads from the current viewport (same working set used by the browser)
reads_df = client.get_reads()
print(f"{len(reads_df)} reads")
reads_df.head()

## 4. Convert to position matrix

The matrix is `(n_reads, region_width)` where each value encodes footprint size:
- **Positive** = footprint size (bp)
- **Negative** = gap/linker size (bp)
- **Zero** = no coverage

In [ ]:
import numpy as np

matrix, index_df = BrowserClient.to_matrix(reads_df)
print(f"Matrix shape: {matrix.shape}")
print(f"Value range: {matrix.min()} to {matrix.max()}")

In [ ]:
# Quick analysis: fraction of positions with nucleosome-sized footprints (>=90bp)
nuc_fraction = (matrix >= 90).mean(axis=1)
print(f"Mean nucleosome occupancy per read: {nuc_fraction.mean():.1%}")
print(f"Range: {nuc_fraction.min():.1%} - {nuc_fraction.max():.1%}")

## 5. Highlight and filter reads in the browser

In [ ]:
# Highlight reads with many footprints
many_fp = reads_df[reads_df["n_footprints"] > 8]["id"].tolist()
client.highlight_reads(many_fp, color="#ff6600", label=">8 footprints")
print(f"Highlighted {len(many_fp)} reads")

In [ ]:
# Filter to show only those reads (hides all others)
client.filter_reads(many_fp)

In [ ]:
# Restore all reads
client.clear_filter()

## 6. Add position markers

In [ ]:
# Add vertical marker lines at positions of interest
region = client.get_region()
mid = (region["start"] + region["end"]) // 2

client.add_markers([
    {"position": mid, "color": "red", "label": "Midpoint"},
    {"position": mid - 500, "color": "blue", "label": "Upstream"},
])

## 7. Push modified reads back to the browser

You can manipulate the matrix in Python and push the result back for visualization.

In [ ]:
# Example: show only reads with a footprint near position 1200
mask = np.abs(matrix[:, min(1200, matrix.shape[1] - 1)]) > 50
client.push_reads(matrix[mask], index_df[mask], label="Footprint @1200")

In [ ]:
# Example: blank out all non-nucleosome footprints
nuc_only = matrix.copy()
nuc_only[(nuc_only > 0) & (nuc_only < 90)] = 0  # keep only nucleosomes
client.push_reads(nuc_only, index_df, label="Nucleosomes only")

In [ ]:
# Restore normal BAM-loaded view
client.clear_pushed_reads()

## 8. Access clustering results

If you've clustered reads in the browser UI, you can retrieve the assignments.

In [ ]:
clusters = client.get_clusters()

if clusters.get("assignments"):
    print(f"{clusters['n_clusters']} clusters, {clusters['n_reads']} reads")
    for k, v in clusters["cluster_counts"].items():
        print(f"  Cluster {k}: {v} reads")
else:
    print("No clustering active. Cluster reads in the browser first.")

## 9. Cleanup

In [ ]:
# Clear all Jupyter-controlled state (highlights, markers, filters, pushed reads)
client.clear_all()